# 9장 실습 ③ — ImageNet 사전학습 모델

**TensorFlow 판**

실무에서는 직접 원천 과제를 학습시키지 않습니다.
**이미 잘 학습된 것을 받아 씁니다.**

> ⚠ 가중치 내려받기가 막힌 환경에서는 건너뜁니다.
> Kaggle이나 Colab에서 돌리시면 받아집니다.

## 9.0 준비

In [ ]:
try:
    import dlbook
except ImportError:
    !pip install -q "dlbook @ git+https://github.com/dhrim/deep-learning-in-one-semester.git"
    import dlbook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import dlbook
from dlbook import data, metrics, plot
from dlbook.data import Split

dlbook.set_seed(42)
plot.use_korean()
print(dlbook.versions())

## 9.1 원천 과제와 목표 과제

**시험 800개·검증 400개를 고정**하고 학습 데이터만 줄입니다.
초고에서는 통째로 줄였다가 시험 12개짜리 표를 만들 뻔했습니다. (7장 §7.6)

In [ ]:
# 원천 과제 — 원·사각형·삼각형 6,000개
xs, ys = data.shapes(6000, seed=42, classes=(0, 1, 2))
src = data.split(xs, ys, val_ratio=0.15, test_ratio=0.15, seed=42)
print("원천 과제:", src)

# 목표 과제 — 십자·마름모. **원천 과제에 없던 도형이다.**
xt, yt = data.shapes(3000, seed=7, classes=(3, 4))

# ★ 시험 800개·검증 400개를 **고정**하고 학습 데이터만 줄인다.
#   (7장 §7.6 — 시험 데이터가 작으면 숫자를 믿을 수 없다.
#    초고에서 통째로 줄였다가 시험 12개짜리 표를 만들 뻔했다.)
x_test, y_test = xt[:800], yt[:800]
x_val, y_val = xt[800:1200], yt[800:1200]
pool_x, pool_y = xt[1200:], yt[1200:]

def target(n):
    """학습 데이터 n개짜리 목표 과제. 시험·검증은 언제나 같다."""
    return Split(pool_x[:n], pool_y[:n], x_val, y_val, x_test, y_test)

fig = plot.image_grid(np.concatenate([xs[:8], xt[:8]]),
                      np.concatenate([ys[:8], yt[:8] + 3]), n=16, cols=8,
                      class_names=list(data.SHAPE_CLASSES))
plt.show()
print("위: 원천 과제의 도형 / 아래: 목표 과제의 도형")

## 9.2 학습 함수 — 여기만 판마다 다릅니다

`train_transfer()` 안의 **①얼린다 → ②분류부만 → ③푼다 → ④낮은 학습률**
순서를 눈여겨보십시오. **처음부터 풀면 배운 것이 망가집니다.**

In [ ]:
import tensorflow as tf

L = tf.keras.layers
SRC_PATH = "ch09_source.keras"

def _cnn(n_classes, seed=42):
    dlbook.set_seed(seed)
    return tf.keras.Sequential([
        L.Input(shape=(28, 28, 1)),
        L.Conv2D(16, 3, activation="relu", padding="same"), L.MaxPooling2D(2),
        L.Conv2D(32, 3, activation="relu", padding="same"), L.MaxPooling2D(2),
        L.Flatten(), L.Dense(64, activation="relu"),
        L.Dense(n_classes, activation="softmax"),
    ])

def _fit(model, sp, epochs, lr):
    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(sp.x_train, sp.y_train, validation_data=(sp.x_val, sp.y_val),
              epochs=dlbook.smoke.epochs(epochs), batch_size=32, verbose=0)
    return metrics.accuracy(sp.y_test, model.predict(sp.x_test, verbose=0).argmax(1))

def train_source(sp):
    """원천 과제를 학습하고 파일로 남긴다."""
    m = _cnn(3)
    acc = _fit(m, sp, 15, 0.001)
    m.save(SRC_PATH)
    return acc

def train_scratch(sp):
    """목표 과제를 처음부터 학습한다."""
    return _fit(_cnn(2), sp, 30, 0.001)

def train_transfer(sp, finetune=False):
    """가져온 특징 추출부로 목표 과제를 학습한다. 얼렸다가 푸는 순서를 지킨다."""
    loaded = tf.keras.models.load_model(SRC_PATH)
    base = tf.keras.Sequential(loaded.layers[:-2])
    base.build((None, 28, 28, 1))
    base.trainable = False
    dlbook.set_seed(42)
    model = tf.keras.Sequential([base, L.Dense(64, activation="relu"),
                                 L.Dense(2, activation="softmax")])
    acc = _fit(model, sp, 30, 0.001)
    if not finetune:
        return acc
    base.trainable = True
    return _fit(model, sp, 15, 0.0001)

def train_pretrained(sp):
    """ImageNet 사전학습 모델."""
    base = tf.keras.applications.MobileNetV2(
        weights="imagenet", include_top=False, input_shape=(96, 96, 3))
    base.trainable = False
    dlbook.set_seed(42)
    model = tf.keras.Sequential([
        L.Input(shape=(28, 28, 1)), L.Resizing(96, 96),
        L.Lambda(lambda t: tf.repeat(t, 3, axis=-1)),
        L.Rescaling(2.0, offset=-1.0), base,
        L.GlobalAveragePooling2D(), L.Dense(64, activation="relu"),
        L.Dense(2, activation="softmax"),
    ])
    return _fit(model, sp, 15, 0.001)

## 9.3 사전학습 모델 가져오기

In [ ]:
# ImageNet 사전학습 모델. 내려받기가 막힌 환경에서는 건너뛴다.
try:
    acc = train_pretrained(target(400))
    dlbook.record("ch09_pretrained_acc", acc)
    print(f"사전학습 모델 전이학습 시험 정확도 {acc:.3f}")
except Exception as e:
    print(f"⚠ 사전학습 가중치를 받지 못했습니다 ({type(e).__name__}).")
    print("  Kaggle이나 Colab에서 돌리시면 받아집니다.")
    print("  §9.3의 도형 실험만으로도 전이학습의 논리는 확인됩니다.")

## 정리

- 실무의 전이학습은 **ImageNet으로 학습된 모델**을 받아 쓰는 것입니다.
- **전처리를 모델에 맞춰야 합니다.** `/255.0` 만 하고 넘기면 안 됩니다.
- 입력 크기와 채널 수도 맞춰야 합니다.
- **ImageNet 정확도가 높다고 내 문제에서도 최고인 것은 아닙니다.**
  두세 개를 골라 **검증 데이터로** 비교하십시오. (7장 §7.6)

### 연습

1. MobileNetV2 대신 ResNet50, VGG16으로 바꿔 비교하십시오.
2. `preprocess_input` 을 **빼고** 돌리면 성능이 얼마나 떨어집니까.
3. 이 결과로 본문 §9.4의 빈 표를 채우십시오.